In [ ]:
!pip install -q pypianoroll
!apt-get install -y -q timidity
!pip install -q midi2audio
!pip install -q pretty_midi

import numpy as np
import pypianoroll
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pretty_midi
from midi2audio import FluidSynth
from IPython.display import Audio
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from tqdm import tqdm
from torch import autograd

In [ ]:
# def play_sample(path):
#     mt = pypianoroll.load(path)
#     pm = mt.to_pretty_midi()
#     pm.write(f"/kaggle/working/temp.mid")
#     !timidity /kaggle/working/temp.mid -Ow -o /kaggle/working/temp.wav -q0
#     display(Audio("/kaggle/working/temp.wav"))

# play_sample("/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/B/Z/B/TRBZBJP12903CB24FB/91901a77e4390431b2b1f245f30f8984.npz")
# play_sample("/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/W/E/D/TRWEDUK12903D0936E/c2e28f932dcc7eb5487587fa6be72d92.npz")

In [ ]:
path = "/kaggle/input/lpd-5-cleansed/lpd_5/lpd_5_cleansed/B/Z/B/TRBZBJP12903CB24FB/91901a77e4390431b2b1f245f30f8984.npz"

multitrack = pypianoroll.load(path)

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True, sharey=True)

track_names = ["Drums", "Bass", "Guitar", "Strings", "Piano"]

for i in range(5):
    roll = multitrack.tracks[i].pianoroll.astype(np.int32)
    axes[i].imshow(roll.T, aspect="auto", origin="lower", cmap="gray_r")
    axes[i].set_ylabel(track_names[i])
    if i == 0:
        axes[i].set_title("Multitrack Pianoroll")

In [ ]:
# Count Number of tracks
# cnt = 0
# for root, _, files in os.walk('/kaggle/input'):
#     for file in files:
#         if file.lower().endswith('.npz'):
#             cnt+=1
# print(cnt)

In [ ]:
class MGDataset(Dataset):
    def __init__(self, root_dir, tracks = 5, t_len = 96*4,pitch_size=128,cond_track_id=0, transform = None):
        self.root_dir = root_dir
        self.transform = transform
        self.t_len = t_len
        self.track_samples = []
        self.humanai_samples = []
        self.pitch_size = pitch_size
        self.cond_track = cond_track_id
        self.tracks= tracks
        for root, _, files in tqdm(os.walk(self.root_dir)):
            for file in files:
                if file.lower().endswith('.npz'):
                    multitrack = pypianoroll.load(os.path.join(root,file))
                    T = multitrack.tracks[0].pianoroll.shape[0]
                    n_samples = T // self.t_len 
                    for i in range(n_samples):
                       self.samples.append((os.path.join(root,file), i *  self.t_len))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, start = self.samples[idx]
        multitrack = pypianoroll.load(path)
        rolls = []
        for track in multitrack.tracks:
            pr = track.pianoroll[start:start+self.t_len]
            if pr.shape[0] < self.t_len:
                pad_width = ((0, self.t_len - pr.shape[0]), (0, 0))
                pr = np.pad(pr, pad_width, mode="constant")
            if pr.shape[1] < self.pitch_size:
                pad_width = ((0, 0), (0, self.pitch_size - pr.shape[1]))
                pr = np.pad(pr, pad_width, mode="constant")
            elif pr.shape[1] > self.pitch_size:
                pr = pr[:, :self.pitch_size]
                
            rolls.append(pr)
        while len(rolls) < self.tracks:
            rolls.append(np.zeros((self.t_len, self.pitch_size)))

        rolls = np.stack(rolls, axis=0)  
        rolls = torch.tensor(rolls, dtype=torch.float32)  / 127.0
        condition_track = rolls[self.cond_track]
        
        target_tracks = torch.cat([rolls[i].unsqueeze(0) for i in range(self.tracks) if i != self.cond_track],dim=0) 
        if self.transform:
            return self.transform((condition_track, target_tracks))
        return condition_track, target_tracks
                    

In [ ]:
class HADiscriminator(nn.Module):
    def __init__(self, inc=5): 
        super(HumanAIDiscriminator, self).__init__()
        
        self.model = nn.Sequential(
            nn.Conv2d(inc, 64, kernel_size=4, stride=2, padding=1), 
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True), 
            
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0),
        )
    
    def forward(self, gen_tracks, cond_track):
        # gen_tracks: (B, 4, T, P)
        # cond_track: (B, 1, T, P)
        x_in = torch.cat([cond_track, gen_tracks], dim=1) 
        out = self.model(x_in)
        return out.view(out.size(0), -1).mean(dim=1, keepdim=True)
        

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, pitch_range=128, time_res=96, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=(3,3), stride=(2,2), padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=(3,3), stride=(2,2), padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=(3,3), stride=(2,2), padding=1),
            nn.ReLU()
        )
        
        self.fc = nn.Linear(256 * (pitch_range//8) * (time_res//8), latent_dim)
        
    def forward(self, x):
        B = x.size(0)
        h = self.conv(x)
        h = h.view(B, -1)
        latent = self.fc(h)  # (B, 128)
        return latent

In [ ]:
class TrackGenerator(nn.Module):
    def __init__(self, input_dim, out_dim, hidden_dim, pitch_range, time_res):
        super().__init__()

        self.pitch_range = pitch_range
        self.time_res = time_res

        self.project = nn.Sequential(
            
            nn.Linear(input_dim, 256 * 6 * 8),
            nn.ReLU()
        )
        
        self.deconv = nn.Sequential(
            
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),  # (128, 12, 16)
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # (64, 24, 32)
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 1, kernel_size=4, stride=2, padding=1),     # (1, 48, 64)
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = self.project(x)                   # (batch, 256*6*8)
        x = x.view(batch_size, 256, 6, 8)     # (batch, 256, 6, 8)
        x = self.deconv(x)                    # (batch, 1, pitch_range, time_res)
        x = F.interpolate(x, size=(self.pitch_range, self.time_res), mode='bilinear', align_corners=False)
        return x.squeeze(1)   

In [ ]:
class BarGenerator(nn.Module) :
    def __init__(self, input_dim, out_dim, hidden_dim, pitch_range, time_res):
        super().__init__()

        self.pitch_range = pitch_range
        self.time_res = time_res

        self.model = nn.Sequential(
            
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim, out_dim),
            nn.ReLU()
        )
        
    def forward(self, x):
        return self.model(x)   # [B, out_dim]

In [ ]:
class MuseGenerator(nn.Module):
    def __init__(self, latent_dim=128,cond_track_ind=0, z_t_dim=32, z_i_dim=32,cond_track_id=0, z_it_dim=32, time_steps=4, num_tracks=4, pitch_range=128, time_res=96, hidden_dim=256):
        super().__init__()

        self.latent_dim = latent_dim
        self.z_t_dim = z_t_dim
        self.z_i_dim = z_i_dim
        self.z_it_dim = z_it_dim
        self.time_steps = time_steps
        self.num_tracks = num_tracks
        self.pitch_range = pitch_range
        self.time_res = time_res
        self.out_dim = pitch_range * time_res
        self.cond_track_id=cond_track_id

        self.bar_generator = BarGenerator(
                input_dim = z_t_dim,
                out_dim = hidden_dim,
                hidden_dim = hidden_dim,
                pitch_range = self.pitch_range,
                time_res = self.time_res
            )

        self.temporal_encoder = TemporalEncoder( 
                latent_dim=self.latent_dim
                pitch_range = self.pitch_range,
                time_res = self.time_res
            )

        self.track_generators = nn.ModuleList([
            TrackGenerator(
                input_dim = z_i_dim + z_it_dim + hidden_dim,
                out_dim = self.out_dim,
                hidden_dim = hidden_dim,
                pitch_range = self.pitch_range,
                time_res = self.time_res
            ) for _ in range(num_tracks)
        ])

    def forward(self, z_t, z_i_list, z_it_list, cond_track):
        batch_size = z_t.size(0)
        
        temporal_context = []

        for t in range(self.time_steps) :
            z_t_step = z_t[:, t, :]
            context_t = self.bar_generator(z_t_step)
            temporal_context.append(context_t)
        
        outputs = []
        temporally_encoded = self.temporal_encoder(cond_track)
        
        for track_idx in range(self.num_tracks):
            z_i = z_i_list[track_idx]
            track_outputs = []
            
            for t in range(self.time_steps):
                z_it = z_it_list[track_idx][t]
                
                combine_ip = torch.cat([z_i, z_it, temporal_context[t], temporally_encoded], dim=1)
                
                pianoroll = self.track_generators[track_idx](combine_ip)

                track_outputs.append(pianoroll)
            track_outputs = torch.stack(track_outputs, dim=1)
            outputs.append(track_outputs)
        
        outputs = torch.stack(outputs, dim=1)
        batch_size, num_tracks, time_steps, pitch_range, time_res = outputs.shape
        
        #  [B, N, T, P, R]
        outputs = outputs.permute(0, 1, 3, 2, 4)  # [B, N, P, T, R]
        outputs = outputs.reshape(batch_size, self.num_tracks, self.pitch_range, self.time_steps * self.time_res)

        return outputs


In [ ]:
# For the critic
def wasserstein_critic_loss(real_output, fake_output):
    return torch.mean(fake_output) - torch.mean(real_output)

# For the generator
def wasserstein_generator_loss(fake_output):
    return -torch.mean(fake_output)

# Gradient Penalty

def gradient_penalty(critic, real_data, fake_data, device=device, lambda_gp=10):
    batch_size = real_data.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    alpha = alpha.expand_as(real_data)

    if real_data.shape != fake_data.shape:
        fake_data = F.interpolate(fake_data, size=real_data.shape[2:], mode='bilinear', align_corners=False)
    
    interpolate = alpha * real_data + (1 - alpha) * fake_data
    interpolate = interpolate.to(device).requires_grad_(True)

    interpolated_score = critic(interpolate)

    gradients = autograd.grad(
        outputs=interpolated_score,
        inputs=interpolate,
        grad_outputs=torch.ones_like(interpolated_score),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)

    gradient_penalty = lambda_gp * ((gradient_norm - 1) ** 2).mean()
    return gradient_penalty

In [ ]:
transform = transforms.Compose([
    transforms.Normalize((0.5,)*5, (0.5,)*5)     
])

root_dir = "/kaggle/input/lpd-5-cleansed/"

dataset = PRDataset(root_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Initialize models

generator = MuseGenerator().to(device)
discriminator = Discriminator().to(device)


opt_G = torch.optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.9))
opt_D = torch.optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.9))